In [ ]:
# CELL 1: Install dependencies
!pip install -q ultralytics opencv-python-headless supervision pandas numpy \
    plotly matplotlib scikit-learn pillow streamlit sqlalchemy openpyxl fpdf2

import os
os.makedirs("TrafficMonitoringSystem/models", exist_ok=True)
os.makedirs("TrafficMonitoringSystem/videos", exist_ok=True)
os.makedirs("TrafficMonitoringSystem/reports", exist_ok=True)
os.makedirs("TrafficMonitoringSystem/charts", exist_ok=True)
os.makedirs("TrafficMonitoringSystem/screenshots", exist_ok=True)
os.chdir("TrafficMonitoringSystem")
print("Project directory ready:", os.getcwd())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 9.1 MB/s eta 0:00:00
Project directory ready: /content/TrafficMonitoringSystem


In [ ]:
%%writefile config.py
"""
Central configuration for the Traffic Monitoring System.
"""

# --- Model settings ---
YOLO_MODEL_PATH = "yolov8n.pt"   # nano model — fast, good enough for demo; swap for yolov8m.pt for accuracy
CONFIDENCE_THRESHOLD = 0.35
IOU_THRESHOLD = 0.45

# COCO class IDs relevant to traffic
VEHICLE_CLASSES = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck",
    1: "Bicycle",
}
PERSON_CLASS = {0: "Person"}
TRAFFIC_LIGHT_CLASS = {9: "Traffic Light"}
STOP_SIGN_CLASS = {11: "Stop Sign"}

ALL_TRACKED_CLASSES = {**VEHICLE_CLASSES, **PERSON_CLASS, **TRAFFIC_LIGHT_CLASS, **STOP_SIGN_CLASS}

# --- Density thresholds (vehicles currently in frame / ROI) ---
DENSITY_THRESHOLDS = {
    "Low": (0, 10),
    "Medium": (11, 25),
    "High": (26, 40),
    "Very High": (41, 10_000),
}
DENSITY_COLORS = {
    "Low": "#2ecc71",       # green
    "Medium": "#f1c40f",    # yellow
    "High": "#e67e22",      # orange
    "Very High": "#e74c3c", # red
}

# --- Signal timing map (seconds) keyed by density band ---
SIGNAL_TIMING = {
    "Low": 20,
    "Medium": 40,
    "High": 60,
    "Very High": 90,
}

# --- Speed estimation ---
SPEED_LIMIT_KMPH = 60
PIXELS_PER_METER = 8.0     # calibrate this per camera/video — see speed_estimator.py notes
FRAME_RATE_ASSUMED = 30

# --- Emergency vehicle keywords (used with a secondary classifier/heuristic — see emergency_vehicle.py) ---
EMERGENCY_LABELS = ["ambulance", "police", "fire truck"]

# --- Database ---
DB_PATH = "traffic.db"

# --- Lanes (as fraction of frame width, left->right boundaries) ---
LANE_BOUNDARIES_FRACTIONS = [0.0, 0.33, 0.66, 1.0]  # 3 lanes

Writing config.py


In [ ]:
%%writefile database.py
"""
SQLite persistence layer for the Traffic Monitoring System.
"""

import sqlite3
from datetime import datetime
from contextlib import contextmanager
from config import DB_PATH


class Database:
    def __init__(self, db_path: str = DB_PATH):
        self.db_path = db_path
        self._init_tables()

    @contextmanager
    def _connect(self):
        conn = sqlite3.connect(self.db_path)
        try:
            yield conn
            conn.commit()
        finally:
            conn.close()

    def _init_tables(self):
        schema = """
        CREATE TABLE IF NOT EXISTS vehicle_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            track_id INTEGER,
            vehicle_type TEXT,
            lane INTEGER,
            timestamp TEXT
        );

        CREATE TABLE IF NOT EXISTS density_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            vehicle_count INTEGER,
            density_level TEXT,
            timestamp TEXT
        );

        CREATE TABLE IF NOT EXISTS speed_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            track_id INTEGER,
            vehicle_type TEXT,
            speed_kmph REAL,
            overspeed INTEGER,
            timestamp TEXT
        );

        CREATE TABLE IF NOT EXISTS violations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            violation_type TEXT,
            track_id INTEGER,
            screenshot_path TEXT,
            timestamp TEXT
        );

        CREATE TABLE IF NOT EXISTS signal_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            density_level TEXT,
            signal_duration INTEGER,
            timestamp TEXT
        );

        CREATE TABLE IF NOT EXISTS prediction_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            horizon_minutes INTEGER,
            predicted_volume REAL,
            timestamp TEXT
        );

        CREATE TABLE IF NOT EXISTS emergency_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            vehicle_type TEXT,
            lane INTEGER,
            timestamp TEXT
        );
        """
        with self._connect() as conn:
            conn.executescript(schema)

    def log_vehicle_event(self, track_id, vehicle_type, lane):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO vehicle_events (track_id, vehicle_type, lane, timestamp) VALUES (?,?,?,?)",
                (track_id, vehicle_type, lane, datetime.now().isoformat()),
            )

    def log_density(self, vehicle_count, density_level):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO density_log (vehicle_count, density_level, timestamp) VALUES (?,?,?)",
                (vehicle_count, density_level, datetime.now().isoformat()),
            )

    def log_speed(self, track_id, vehicle_type, speed_kmph, overspeed):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO speed_log (track_id, vehicle_type, speed_kmph, overspeed, timestamp) VALUES (?,?,?,?,?)",
                (track_id, vehicle_type, speed_kmph, int(overspeed), datetime.now().isoformat()),
            )

    def log_violation(self, violation_type, track_id, screenshot_path):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO violations (violation_type, track_id, screenshot_path, timestamp) VALUES (?,?,?,?)",
                (violation_type, track_id, screenshot_path, datetime.now().isoformat()),
            )

    def log_signal(self, density_level, signal_duration):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO signal_log (density_level, signal_duration, timestamp) VALUES (?,?,?)",
                (density_level, signal_duration, datetime.now().isoformat()),
            )

    def log_emergency(self, vehicle_type, lane):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO emergency_log (vehicle_type, lane, timestamp) VALUES (?,?,?)",
                (vehicle_type, lane, datetime.now().isoformat()),
            )

    def fetch_all(self, table: str, limit: int = 500):
        with self._connect() as conn:
            cur = conn.execute(f"SELECT * FROM {table} ORDER BY id DESC LIMIT ?", (limit,))
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
        return cols, rows

Writing database.py


In [ ]:
%%writefile detector.py
"""
YOLOv8-based object detector wrapper.
"""

from ultralytics import YOLO
from config import YOLO_MODEL_PATH, CONFIDENCE_THRESHOLD, IOU_THRESHOLD, ALL_TRACKED_CLASSES


class VehicleDetector:
    def __init__(self, model_path: str = YOLO_MODEL_PATH):
        self.model = YOLO(model_path)
        self.class_names = ALL_TRACKED_CLASSES

    def detect(self, frame):
        """
        Runs detection on a single frame.
        Returns list of dicts: {bbox, conf, class_id, label}
        """
        results = self.model.predict(
            frame,
            conf=CONFIDENCE_THRESHOLD,
            iou=IOU_THRESHOLD,
            classes=list(self.class_names.keys()),
            verbose=False,
        )[0]

        detections = []
        for box in results.boxes:
            cls_id = int(box.cls[0])
            if cls_id not in self.class_names:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            detections.append({
                "bbox": (int(x1), int(y1), int(x2), int(y2)),
                "conf": conf,
                "class_id": cls_id,
                "label": self.class_names[cls_id],
            })
        return detections

Writing detector.py


In [ ]:
%%writefile tracker.py
"""
Multi-object tracking wrapper using the 'supervision' library's ByteTrack
implementation (avoids needing a separate ByteTrack repo/install).
"""

import numpy as np
import supervision as sv


class VehicleTracker:
    def __init__(self, frame_rate: int = 30):
        self.tracker = sv.ByteTrack(frame_rate=frame_rate)
        self.track_history = {}   # track_id -> list of (cx, cy, frame_idx)

    def update(self, detections: list, frame_idx: int):
        if not detections:
            empty = sv.Detections.empty()
            tracked = self.tracker.update_with_detections(empty)
            return []

        xyxy = np.array([d["bbox"] for d in detections], dtype=float)
        confidence = np.array([d["conf"] for d in detections], dtype=float)
        class_id = np.array([d["class_id"] for d in detections], dtype=int)

        sv_detections = sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_id)
        tracked = self.tracker.update_with_detections(sv_detections)

        results = []
        for i in range(len(tracked)):
            x1, y1, x2, y2 = tracked.xyxy[i]
            track_id = int(tracked.tracker_id[i])
            cls_id = int(tracked.class_id[i])
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

            self.track_history.setdefault(track_id, []).append((cx, cy, frame_idx))
            if len(self.track_history[track_id]) > 50:
                self.track_history[track_id].pop(0)

            results.append({
                "track_id": track_id,
                "bbox": (int(x1), int(y1), int(x2), int(y2)),
                "class_id": cls_id,
                "center": (cx, cy),
                "path": self.track_history[track_id],
            })
        return results

    def get_history(self, track_id):
        return self.track_history.get(track_id, [])

Writing tracker.py


In [ ]:
%%writefile density.py
"""
Traffic density classification and lane assignment.
"""

from config import DENSITY_THRESHOLDS, DENSITY_COLORS, LANE_BOUNDARIES_FRACTIONS


def classify_density(vehicle_count: int) -> str:
    for level, (low, high) in DENSITY_THRESHOLDS.items():
        if low <= vehicle_count <= high:
            return level
    return "Very High"


def density_color(level: str) -> str:
    return DENSITY_COLORS.get(level, "#ffffff")


def assign_lane(cx: float, frame_width: int) -> int:
    """
    Returns lane index (1-based) given a vehicle's x-center and frame width.
    """
    fraction = cx / frame_width
    boundaries = LANE_BOUNDARIES_FRACTIONS
    for i in range(len(boundaries) - 1):
        if boundaries[i] <= fraction < boundaries[i + 1]:
            return i + 1
    return len(boundaries) - 1


def count_per_lane(tracked_objects: list, frame_width: int) -> dict:
    counts = {}
    for obj in tracked_objects:
        lane = assign_lane(obj["center"][0], frame_width)
        counts[lane] = counts.get(lane, 0) + 1
    return counts

Writing density.py


In [ ]:
%%writefile signal_controller.py
"""
Smart signal timing logic, with emergency vehicle override.
"""

import time
from config import SIGNAL_TIMING


class SignalController:
    def __init__(self):
        self.current_density_level = "Low"
        self.current_duration = SIGNAL_TIMING["Low"]
        self.timer_start = time.time()
        self.emergency_active = False
        self.emergency_lane = None

    def update(self, density_level: str):
        """Call once per density evaluation cycle (not every frame)."""
        if self.emergency_active:
            return  # emergency override holds priority

        self.current_density_level = density_level
        self.current_duration = SIGNAL_TIMING.get(density_level, SIGNAL_TIMING["Low"])
        self.timer_start = time.time()

    def trigger_emergency(self, lane: int):
        self.emergency_active = True
        self.emergency_lane = lane
        self.timer_start = time.time()
        self.current_duration = 30  # fixed clearance window, tune as needed

    def clear_emergency(self):
        self.emergency_active = False
        self.emergency_lane = None

    def time_remaining(self) -> int:
        elapsed = time.time() - self.timer_start
        remaining = max(0, int(self.current_duration - elapsed))
        return remaining

    def is_green_lane(self, lane: int) -> bool:
        if self.emergency_active:
            return lane == self.emergency_lane
        return True  # simplified: single-approach demo; extend to per-lane phase logic for a full intersection

    def status(self) -> dict:
        return {
            "density_level": self.current_density_level,
            "duration": self.current_duration,
            "remaining": self.time_remaining(),
            "emergency_active": self.emergency_active,
            "emergency_lane": self.emergency_lane,
        }

Writing signal_controller.py


In [ ]:
%%writefile speed_estimator.py
"""
Simple speed estimation from tracked pixel displacement.

IMPORTANT CALIBRATION NOTE:
Pixel-to-meter conversion (PIXELS_PER_METER) must be calibrated per camera
by measuring a known real-world distance (e.g. lane width ~3.5m, or distance
between two road markings) in pixels in your actual video. The default value
is a placeholder and will NOT give accurate real-world speeds until calibrated.
"""

from config import PIXELS_PER_METER, FRAME_RATE_ASSUMED, SPEED_LIMIT_KMPH


def estimate_speed(path: list, frame_rate: int = FRAME_RATE_ASSUMED,
                    pixels_per_meter: float = PIXELS_PER_METER) -> float:
    """
    path: list of (cx, cy, frame_idx) tuples from tracker history.
    Returns speed in km/h based on displacement over the last N frames.
    """
    if len(path) < 2:
        return 0.0

    (x1, y1, f1) = path[0]
    (x2, y2, f2) = path[-1]
    frame_diff = f2 - f1
    if frame_diff <= 0:
        return 0.0

    pixel_dist = ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5
    meters = pixel_dist / pixels_per_meter
    seconds = frame_diff / frame_rate
    if seconds <= 0:
        return 0.0

    mps = meters / seconds
    kmph = mps * 3.6
    return round(kmph, 1)


def is_overspeed(speed_kmph: float, limit: float = SPEED_LIMIT_KMPH) -> bool:
    return speed_kmph > limit

Writing speed_estimator.py


In [ ]:
%%writefile emergency_vehicle.py
"""
Emergency vehicle detection heuristic.

NOTE: The pretrained COCO YOLOv8 model does NOT have "ambulance" / "police" /
"fire truck" classes — it only knows generic "car"/"truck"/"bus". A real
system needs a fine-tuned classifier (e.g. a second-stage CNN on cropped
detections) trained on labeled emergency vehicle images. This module provides
the integration point/interface for that model; wire in a trained classifier
here when you have one.
"""

from config import EMERGENCY_LABELS


class EmergencyVehicleDetector:
    def __init__(self, classifier=None):
        """
        classifier: optional callable(cropped_image) -> label:str
        If None, this module is a no-op stub (always returns False) —
        replace with a trained model for real emergency detection.
        """
        self.classifier = classifier

    def check(self, cropped_image) -> str | None:
        if self.classifier is None:
            return None
        label = self.classifier(cropped_image)
        if label and label.lower() in EMERGENCY_LABELS:
            return label
        return None

Writing emergency_vehicle.py


In [ ]:
%%writefile utils.py
"""
Shared drawing / helper utilities.
"""

import cv2

COLOR_MAP = {
    "Car": (46, 204, 113),
    "Motorcycle": (241, 196, 15),
    "Bus": (230, 126, 34),
    "Truck": (231, 76, 60),
    "Bicycle": (155, 89, 182),
    "Person": (52, 152, 219),
    "Traffic Light": (149, 165, 166),
    "Stop Sign": (192, 57, 43),
}


def draw_box(frame, bbox, label, track_id=None, extra_text=None):
    x1, y1, x2, y2 = bbox
    color = COLOR_MAP.get(label, (255, 255, 255))
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    text = f"{label}"
    if track_id is not None:
        text += f" ID:{track_id}"
    if extra_text:
        text += f" {extra_text}"

    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, text, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return frame


def draw_hud(frame, fps, density_level, signal_status):
    h, w = frame.shape[:2]
    overlay_lines = [
        f"FPS: {fps:.1f}",
        f"Density: {density_level}",
        f"Signal: {signal_status['duration']}s (remaining {signal_status['remaining']}s)",
    ]
    if signal_status.get("emergency_active"):
        overlay_lines.append(f"EMERGENCY MODE - Lane {signal_status['emergency_lane']}")

    y = 25
    for line in overlay_lines:
        cv2.putText(frame, line, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        y += 25
    return frame

Writing utils.py


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload a short traffic clip (mp4)
video_path = list(uploaded.keys())[0]
print("Using video:", video_path)

Saving istockphoto-2278287172-640_adpp_is.mp4 to istockphoto-2278287172-640_adpp_is.mp4
Using video: istockphoto-2278287172-640_adpp_is.mp4


In [ ]:
import cv2
import time
from detector import VehicleDetector
from tracker import VehicleTracker
from density import classify_density, count_per_lane, assign_lane
from signal_controller import SignalController
from speed_estimator import estimate_speed, is_overspeed
from database import Database
from utils import draw_box, draw_hud
from config import VEHICLE_CLASSES

detector = VehicleDetector()
tracker = VehicleTracker()
signal_ctrl = SignalController()
db = Database()

cap = cv2.VideoCapture(video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_video = cap.get(cv2.CAP_PROP_FPS) or 30

out_path = "output_annotated.mp4"
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_video, (frame_width, frame_height))

frame_idx = 0
counted_ids = set()
DENSITY_EVAL_INTERVAL = 15  # frames between density/signal re-evaluations

prev_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    detections = detector.detect(frame)
    tracked = tracker.update(detections, frame_idx)

    # --- Draw boxes + speed + logging ---
    for obj in tracked:
        label = VEHICLE_CLASSES.get(obj["class_id"], None)
        if label is None:
            continue  # skip person/traffic-light/stop-sign boxes for vehicle-only logic below (drawn separately if desired)

        speed = estimate_speed(obj["path"], frame_rate=fps_video)
        overspeed = is_overspeed(speed)
        extra = f"{speed:.0f}km/h" + (" OVERSPEED" if overspeed else "")
        draw_box(frame, obj["bbox"], label, obj["track_id"], extra)

        if obj["track_id"] not in counted_ids:
            counted_ids.add(obj["track_id"])
            lane = assign_lane(obj["center"][0], frame_width)
            db.log_vehicle_event(obj["track_id"], label, lane)

        if speed > 0:
            db.log_speed(obj["track_id"], label, speed, overspeed)

    # --- Density + signal update every N frames ---
    if frame_idx % DENSITY_EVAL_INTERVAL == 0:
        vehicle_count = len([o for o in tracked if o["class_id"] in VEHICLE_CLASSES])
        level = classify_density(vehicle_count)
        signal_ctrl.update(level)
        db.log_density(vehicle_count, level)
        db.log_signal(level, signal_ctrl.current_duration)

    # --- HUD ---
    now = time.time()
    fps_display = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now
    draw_hud(frame, fps_display, signal_ctrl.current_density_level, signal_ctrl.status())

    writer.write(frame)

    if frame_idx % 50 == 0:
        print(f"Processed frame {frame_idx}")

cap.release()
writer.release()
print("Done. Annotated video saved to:", out_path)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Processed frame 50
Processed frame 100
Processed frame 150
Processed frame 200
Processed frame 250
Processed frame 300
Done. Annotated video saved to: output_annotated.mp4


In [ ]:
import pandas as pd
import plotly.express as px

cols, rows = db.fetch_all("vehicle_events")
df = pd.DataFrame(rows, columns=cols)

if not df.empty:
    fig = px.pie(df, names="vehicle_type", title="Vehicle Type Distribution")
    fig.show()

    # Get the value counts and reset the index
    lane_counts_df = df["lane"].value_counts().reset_index()
    # Rename the columns to be explicit:
    # 'index' column contains the lane numbers
    # 'lane' column (from value_counts()) contains the vehicle counts
    lane_counts_df.columns = ['lane_number', 'vehicle_count']

    fig2 = px.bar(lane_counts_df, x="lane_number", y="vehicle_count",
                   labels={"lane_number": "Lane", "vehicle_count": "Vehicle Count"}, title="Vehicles per Lane")
    fig2.show()
else:
    print("No vehicle events logged yet — run Cell 12 on a video first.")

In [ ]:
%%writefile app.py
"""
AI-Based Intelligent Traffic Monitoring and Smart Traffic Signal Management System
Main Streamlit Dashboard

Currently running on dummy/simulated data. Live AI modules (detector.py,
tracker.py, density.py, etc.) will be wired into the "Live Detection" page
and the dummy-data generators below will be swapped for database.py reads
in a later pass.
"""

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import time
import random

# ============================================================
# PAGE CONFIG
# ============================================================
st.set_page_config(
    page_title="Smart Traffic Control System",
    page_icon="🚦",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ============================================================
# DARK THEME CSS
# ============================================================
CUSTOM_CSS = """
<style>
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}

    .stApp {
        background: linear-gradient(180deg, #0b0f19 0%, #0d1220 100%);
        color: #e6e9f0;
    }

    section[data-testid="stSidebar"] {
        background: #0a0e17;
        border-right: 1px solid #1c2333;
    }
    section[data-testid="stSidebar"] .stRadio label {
        font-size: 15px;
    }

    h1, h2, h3, h4 {
        color: #f2f4f8 !important;
        font-family: 'Segoe UI', sans-serif;
    }

    /* KPI card */
    .kpi-card {
        background: linear-gradient(145deg, #131a2b, #0f1524);
        border: 1px solid #232b40;
        border-radius: 14px;
        padding: 18px 20px;
        box-shadow: 0 4px 14px rgba(0,0,0,0.35);
        transition: transform 0.15s ease;
    }
    .kpi-card:hover {
        transform: translateY(-3px);
        border-color: #3b82f6;
    }
    .kpi-label {
        font-size: 13px;
        color: #8b94ab;
        text-transform: uppercase;
        letter-spacing: 0.5px;
        margin-bottom: 6px;
    }
    .kpi-value {
        font-size: 30px;
        font-weight: 700;
        color: #f2f4f8;
    }
    .kpi-icon {
        font-size: 26px;
        margin-bottom: 4px;
    }
    .kpi-sub {
        font-size: 12px;
        color: #5fd68a;
        margin-top: 4px;
    }

    .status-pill {
        display: inline-block;
        padding: 6px 16px;
        border-radius: 20px;
        font-weight: 600;
        font-size: 14px;
    }

    .section-title {
        font-size: 20px;
        font-weight: 700;
        margin-top: 10px;
        margin-bottom: 14px;
        color: #f2f4f8;
        border-left: 4px solid #3b82f6;
        padding-left: 10px;
    }

    .panel {
        background: #0f1524;
        border: 1px solid #1c2333;
        border-radius: 14px;
        padding: 18px;
    }

    div[data-testid="stMetricValue"] {
        color: #f2f4f8;
    }

    .stDataFrame {
        border-radius: 10px;
        overflow: hidden;
    }

    .footer-tag {
        text-align: center;
        color: #4b5470;
        font-size: 12px;
        margin-top: 40px;
    }
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

# ============================================================
# PLOTLY DARK TEMPLATE
# ============================================================
PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#0f1524"
PAPER_BG = "#0f1524"
GRID_COLOR = "#1c2333"

def style_fig(fig, height=350):
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        plot_bgcolor=PLOT_BG,
        paper_bgcolor=PAPER_BG,
        font=dict(color="#c8cede", family="Segoe UI"),
        height=height,
        margin=dict(l=10, r=10, t=45, b=10),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
    fig.update_xaxes(gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR)
    fig.update_yaxes(gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR)
    return fig

DENSITY_COLORS = {
    "Low": "#2ecc71",
    "Medium": "#f1c40f",
    "High": "#e67e22",
    "Very High": "#e74c3c",
}
VEHICLE_COLORS = {
    "Car": "#3b82f6",
    "Bike": "#f1c40f",
    "Bus": "#e67e22",
    "Truck": "#e74c3c",
}

# ============================================================
# DUMMY DATA GENERATORS
# (These are the integration points — later swapped for database.py reads)
# ============================================================

@st.cache_data(ttl=5)
def generate_kpis():
    cars = random.randint(120, 220)
    bikes = random.randint(60, 140)
    bus = random.randint(5, 25)
    truck = random.randint(10, 40)
    total = cars + bikes + bus + truck
    avg_speed = round(random.uniform(28, 58), 1)
    density = random.choice(["Low", "Medium", "High", "Very High"])
    signal = random.choice(["GREEN", "YELLOW", "RED"])
    emergency = random.choice([False, False, False, False, True])
    return {
        "total": total, "cars": cars, "bikes": bikes, "bus": bus, "truck": truck,
        "avg_speed": avg_speed, "density": density, "signal": signal, "emergency": emergency,
    }

@st.cache_data(ttl=30)
def generate_hourly_traffic():
    hours = [f"{h:02d}:00" for h in range(24)]
    base = [30, 20, 15, 12, 18, 35, 80, 150, 190, 160, 130, 140,
            150, 145, 135, 150, 175, 200, 210, 170, 120, 90, 60, 40]
    values = [max(5, b + random.randint(-20, 20)) for b in base]
    return pd.DataFrame({"Hour": hours, "Vehicles": values})

@st.cache_data(ttl=30)
def generate_density_trend():
    now = datetime.now()
    times = [(now - timedelta(minutes=5 * i)).strftime("%H:%M") for i in range(30)][::-1]
    counts = np.clip(np.cumsum(np.random.randn(30)) * 3 + 25, 2, 60).astype(int)
    levels = []
    for c in counts:
        if c <= 10: levels.append("Low")
        elif c <= 25: levels.append("Medium")
        elif c <= 40: levels.append("High")
        else: levels.append("Very High")
    return pd.DataFrame({"Time": times, "VehicleCount": counts, "Density": levels})

@st.cache_data(ttl=30)
def generate_speed_data():
    now = datetime.now()
    times = [(now - timedelta(minutes=2 * i)).strftime("%H:%M") for i in range(20)][::-1]
    speeds = np.clip(np.random.normal(45, 10, 20), 15, 90)
    return pd.DataFrame({"Time": times, "AvgSpeed": speeds.round(1)})

@st.cache_data(ttl=30)
def generate_lane_counts():
    lanes = ["Lane 1", "Lane 2", "Lane 3"]
    counts = [random.randint(15, 60) for _ in lanes]
    return pd.DataFrame({"Lane": lanes, "VehicleCount": counts})

@st.cache_data(ttl=30)
def generate_vehicle_distribution(kpis):
    return pd.DataFrame({
        "Type": ["Car", "Bike", "Bus", "Truck"],
        "Count": [kpis["cars"], kpis["bikes"], kpis["bus"], kpis["truck"]],
    })

@st.cache_data(ttl=60)
def generate_violation_log():
    types = ["Red Light Violation", "Wrong Lane", "Illegal Parking", "Overspeed"]
    rows = []
    now = datetime.now()
    for i in range(15):
        rows.append({
            "ID": f"V{1000+i}",
            "Type": random.choice(types),
            "Vehicle ID": random.randint(1, 300),
            "Time": (now - timedelta(minutes=random.randint(1, 500))).strftime("%Y-%m-%d %H:%M:%S"),
            "Location": random.choice(["Junction A", "Junction B", "Main St", "Ring Rd"]),
        })
    return pd.DataFrame(rows).sort_values("Time", ascending=False)

@st.cache_data(ttl=60)
def generate_prediction_data():
    now = datetime.now()
    future_times = [(now + timedelta(minutes=5 * i)).strftime("%H:%M") for i in range(1, 7)]
    predicted = np.clip(np.cumsum(np.random.randn(6)) * 4 + 90, 20, 220).astype(int)
    return pd.DataFrame({"Time": future_times, "PredictedVolume": predicted})

# ============================================================
# SIDEBAR NAVIGATION
# ============================================================
with st.sidebar:
    st.markdown("## 🚦 TrafficAI")
    st.markdown("###### Smart Traffic Control System")
    st.markdown("---")
    page = st.radio(
        "Navigation",
        ["📊 Dashboard", "🎥 Live Detection", "📈 Analytics", "📄 Reports", "🔮 Prediction", "⚙️ Settings", "ℹ️ About"],
        label_visibility="collapsed",
    )
    st.markdown("---")
    st.markdown("###### System Status")
    st.markdown("🟢 **Online** — Simulated Feed")
    st.caption(f"Last sync: {datetime.now().strftime('%H:%M:%S')}")
    st.markdown("---")
    auto_refresh = st.checkbox("Auto-refresh (5s)", value=False)

# ============================================================
# REUSABLE KPI CARD COMPONENT
# ============================================================
def kpi_card(col, icon, label, value, sub=None, color="#3b82f6"):
    with col:
        st.markdown(f"""
        <div class="kpi-card">
            <div class="kpi-icon">{icon}</div>
            <div class="kpi-label">{label}</div>
            <div class="kpi-value" style="color:{color}">{value}</div>
            {f'<div class="kpi-sub">{sub}</div>' if sub else ''}
        </div>
        """, unsafe_allow_html=True)


# ============================================================
# PAGE: DASHBOARD
# ============================================================
if page == "📊 Dashboard":
    st.markdown("# Traffic Control Dashboard")
    st.caption(f"Real-time overview • {datetime.now().strftime('%A, %d %B %Y — %H:%M:%S')}")

    kpis = generate_kpis()

    st.markdown('<div class="section-title">Live Metrics</div>', unsafe_allow_html=True)
    c1, c2, c3, c4, c5 = st.columns(5)
    kpi_card(c1, "🚗", "Total Vehicles", kpis["total"], "Last 5 min", "#3b82f6")
    kpi_card(c2, "🚘", "Cars", kpis["cars"], color="#5fa8f5")
    kpi_card(c3, "🚌", "Bus", kpis["bus"], color="#e67e22")
    kpi_card(c4, "🚚", "Truck", kpis["truck"], color="#e74c3c")
    kpi_card(c5, "🏍️", "Bike", kpis["bikes"], color="#f1c40f")

    st.write("")
    c6, c7, c8, c9 = st.columns(4)
    kpi_card(c6, "⚡", "Average Speed", f'{kpis["avg_speed"]} km/h', color="#2ecc71")

    density_color = DENSITY_COLORS[kpis["density"]]
    with c7:
        st.markdown(f"""
        <div class="kpi-card">
            <div class="kpi-icon">🚦</div>
            <div class="kpi-label">Current Density</div>
            <div class="status-pill" style="background:{density_color}22; color:{density_color}; border:1px solid {density_color};">
                {kpis["density"]}
            </div>
        </div>
        """, unsafe_allow_html=True)

    signal_color = {"GREEN": "#2ecc71", "YELLOW": "#f1c40f", "RED": "#e74c3c"}[kpis["signal"]]
    with c8:
        st.markdown(f"""
        <div class="kpi-card">
            <div class="kpi-icon">🚥</div>
            <div class="kpi-label">Signal Status</div>
            <div class="status-pill" style="background:{signal_color}22; color:{signal_color}; border:1px solid {signal_color};">
                {kpis["signal"]}
            </div>
        </div>
        """, unsafe_allow_html=True)

    emergency_text = "ACTIVE 🚨" if kpis["emergency"] else "NORMAL"
    emergency_color = "#e74c3c" if kpis["emergency"] else "#2ecc71"
    with c9:
        st.markdown(f"""
        <div class="kpi-card">
            <div class="kpi-icon">🆘</div>
            <div class="kpi-label">Emergency Status</div>
            <div class="status-pill" style="background:{emergency_color}22; color:{emergency_color}; border:1px solid {emergency_color};">
                {emergency_text}
            </div>
        </div>
        """, unsafe_allow_html=True)

    st.write("")
    st.markdown('<div class="section-title">Traffic Overview</div>', unsafe_allow_html=True)

    col_a, col_b = st.columns([1, 1])
    with col_a:
        dist_df = generate_vehicle_distribution(kpis)
        fig = px.pie(dist_df, names="Type", values="Count", hole=0.55,
                     color="Type", color_discrete_map=VEHICLE_COLORS,
                     title="Vehicle Distribution")
        fig.update_traces(textinfo="percent+label")
        st.plotly_chart(style_fig(fig), use_container_width=True)

    with col_b:
        hourly_df = generate_hourly_traffic()
        fig2 = px.area(hourly_df, x="Hour", y="Vehicles", title="Hourly Traffic Volume")
        fig2.update_traces(line_color="#3b82f6", fillcolor="rgba(59,130,246,0.25)")
        st.plotly_chart(style_fig(fig2), use_container_width=True)

    col_c, col_d = st.columns([1, 1])
    with col_c:
        density_df = generate_density_trend()
        fig3 = px.line(density_df, x="Time", y="VehicleCount", title="Density Trend (last 2.5 hrs)",
                        markers=True)
        fig3.update_traces(line_color="#e67e22")
        st.plotly_chart(style_fig(fig3), use_container_width=True)

    with col_d:
        lane_df = generate_lane_counts()
        fig4 = px.bar(lane_df, x="Lane", y="VehicleCount", title="Lane-wise Vehicle Count",
                       color="VehicleCount", color_continuous_scale="Blues")
        st.plotly_chart(style_fig(fig4), use_container_width=True)


# ============================================================
# PAGE: LIVE DETECTION
# ============================================================
elif page == "🎥 Live Detection":
    st.markdown("# Live Detection Feed")
    st.caption("Simulated feed — connects to detector.py + tracker.py in the AI integration pass")

    col_video, col_info = st.columns([2, 1])

    with col_video:
        st.markdown('<div class="panel">', unsafe_allow_html=True)
        st.markdown("#### 📹 Camera Feed — Junction A")
        st.image(
            "https://placehold.co/960x540/0f1524/3b82f6?text=Live+Camera+Feed+%28Simulated%29",
            use_container_width=True,
        )
        b1, b2, b3 = st.columns(3)
        b1.button("▶️ Start Feed", use_container_width=True)
        b2.button("⏸️ Pause", use_container_width=True)
        b3.button("⏹️ Stop", use_container_width=True)
        st.markdown('</div>', unsafe_allow_html=True)

    with col_info:
        kpis = generate_kpis()
        st.markdown('<div class="panel">', unsafe_allow_html=True)
        st.markdown("#### Frame Stats")
        st.metric("FPS", f"{random.randint(24, 30)}")
        st.metric("Vehicles in Frame", kpis["total"] % 40 + 5)
        st.metric("Current Density", kpis["density"])
        st.metric("Signal Timer", f"{random.randint(5, 60)}s")
        if kpis["emergency"]:
            st.error("🚨 Emergency Vehicle Detected — Lane 2")
        else:
            st.success("✅ No Emergency Vehicles")
        st.markdown('</div>', unsafe_allow_html=True)

    st.write("")
    st.markdown('<div class="section-title">Detected Objects (current frame)</div>', unsafe_allow_html=True)
    obj_rows = []
    for i in range(random.randint(6, 12)):
        obj_rows.append({
            "Track ID": random.randint(100, 999),
            "Type": random.choice(["Car", "Bike", "Bus", "Truck", "Person"]),
            "Lane": random.choice([1, 2, 3]),
            "Speed (km/h)": round(random.uniform(10, 75), 1),
            "Confidence": f"{random.uniform(0.75, 0.98):.2f}",
        })
    st.dataframe(pd.DataFrame(obj_rows), use_container_width=True, hide_index=True)


# ============================================================
# PAGE: ANALYTICS
# ============================================================
elif page == "📈 Analytics":
    st.markdown("# Traffic Analytics")
    st.caption("Historical trends and breakdowns")

    kpis = generate_kpis()

    col1, col2 = st.columns(2)
    with col1:
        dist_df = generate_vehicle_distribution(kpis)
        fig = px.bar(dist_df, x="Type", y="Count", color="Type",
                     color_discrete_map=VEHICLE_COLORS, title="Vehicle Type Distribution")
        st.plotly_chart(style_fig(fig), use_container_width=True)

    with col2:
        hourly_df = generate_hourly_traffic()
        fig2 = px.line(hourly_df, x="Hour", y="Vehicles", title="Hourly Traffic Pattern", markers=True)
        fig2.update_traces(line_color="#3b82f6")
        peak_hour = hourly_df.loc[hourly_df["Vehicles"].idxmax(), "Hour"]
        st.plotly_chart(style_fig(fig2), use_container_width=True)
        st.info(f"📌 Peak Hour: **{peak_hour}**")

    col3, col4 = st.columns(2)
    with col3:
        density_df = generate_density_trend()
        fig3 = px.bar(density_df, x="Time", y="VehicleCount", color="Density",
                       color_discrete_map=DENSITY_COLORS, title="Density Trend")
        st.plotly_chart(style_fig(fig3), use_container_width=True)

    with col4:
        speed_df = generate_speed_data()
        fig4 = px.line(speed_df, x="Time", y="AvgSpeed", title="Average Speed Over Time", markers=True)
        fig4.add_hline(y=60, line_dash="dash", line_color="#e74c3c",
                        annotation_text="Speed Limit (60 km/h)")
        fig4.update_traces(line_color="#2ecc71")
        st.plotly_chart(style_fig(fig4), use_container_width=True)

    st.markdown('<div class="section-title">Lane-wise Analysis & Heatmap</div>', unsafe_allow_html=True)
    col5, col6 = st.columns(2)
    with col5:
        lane_df = generate_lane_counts()
        fig5 = px.bar(lane_df, x="Lane", y="VehicleCount", color="Lane", title="Lane-wise Count")
        st.plotly_chart(style_fig(fig5), use_container_width=True)

    with col6:
        heat_data = np.random.randint(5, 60, size=(7, 24))
        days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        hours = [f"{h}:00" for h in range(24)]
        fig6 = go.Figure(data=go.Heatmap(z=heat_data, x=hours, y=days, colorscale="Blues"))
        fig6.update_layout(title="Weekly Traffic Heatmap")
        st.plotly_chart(style_fig(fig6), use_container_width=True)


# ============================================================
# PAGE: REPORTS
# ============================================================
elif page == "📄 Reports":
    st.markdown("# Reports & Violations")
    st.caption("Generate and export traffic reports")

    tab1, tab2 = st.tabs(["📋 Violation Log", "📦 Export Reports"])

    with tab1:
        viol_df = generate_violation_log()
        f1, f2 = st.columns([1, 3])
        with f1:
            type_filter = st.selectbox("Filter by type", ["All"] + list(viol_df["Type"].unique()))
        if type_filter != "All":
            viol_df = viol_df[viol_df["Type"] == type_filter]
        st.dataframe(viol_df, use_container_width=True, hide_index=True)

        c1, c2, c3 = st.columns(3)
        c1.metric("Total Violations (24h)", len(viol_df))
        c2.metric("Most Common", viol_df["Type"].mode()[0] if not viol_df.empty else "-")
        c3.metric("Locations Monitored", viol_df["Location"].nunique() if not viol_df.empty else 0)

    with tab2:
        st.markdown("#### Export current data")
        colf1, colf2, colf3 = st.columns(3)
        with colf1:
            st.markdown("**CSV Export**")
            csv_data = generate_violation_log().to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download CSV", csv_data, "traffic_report.csv", "text/csv",
                                use_container_width=True)
        with colf2:
            st.markdown("**Excel Export**")
            st.button("⬇️ Download Excel", use_container_width=True,
                       help="Wired to analytics.py export function in the AI integration pass")
        with colf3:
            st.markdown("**PDF Export**")
            st.button("⬇️ Download PDF", use_container_width=True,
                       help="Wired to analytics.py export function in the AI integration pass")

        st.markdown("---")
        st.markdown("#### Report Preview")
        summary_df = pd.DataFrame({
            "Metric": ["Total Vehicles", "Total Violations", "Average Speed", "Peak Hour", "Most Congested Lane"],
            "Value": [random.randint(1000, 3000), random.randint(20, 80),
                      f"{random.uniform(35, 55):.1f} km/h", "18:00", "Lane 2"],
        })
        st.table(summary_df)


# ============================================================
# PAGE: PREDICTION
# ============================================================
elif page == "🔮 Prediction":
    st.markdown("# Traffic Prediction")
    st.caption("ML-based short-term forecasting — will connect to prediction.py (Random Forest / XGBoost / Linear Regression)")

    col1, col2 = st.columns([2, 1])
    with col1:
        pred_df = generate_prediction_data()
        fig = px.line(pred_df, x="Time", y="PredictedVolume", markers=True,
                       title="Predicted Traffic Volume (Next 30 min)")
        fig.update_traces(line_color="#8b5cf6")
        st.plotly_chart(style_fig(fig), use_container_width=True)

    with col2:
        st.markdown('<div class="panel">', unsafe_allow_html=True)
        st.markdown("#### Model Info")
        model_choice = st.selectbox("Active Model", ["Random Forest", "XGBoost", "Linear Regression"])
        st.metric("Model Accuracy (R²)", f"{random.uniform(0.82, 0.95):.2f}")
        st.metric("Next 5 min", f"{random.randint(60, 200)} vehicles")
        st.metric("Next 10 min", f"{random.randint(60, 200)} vehicles")
        recommended = random.choice(["20s", "40s", "60s", "90s"])
        st.metric("Recommended Signal Timing", recommended)
        st.markdown('</div>', unsafe_allow_html=True)

    st.markdown('<div class="section-title">Historical vs Predicted</div>', unsafe_allow_html=True)
    hourly_df = generate_hourly_traffic()
    hist_hours = hourly_df["Hour"].tolist()[-6:]
    hist_vals = hourly_df["Vehicles"].tolist()[-6:]
    combined = pd.DataFrame({
        "Time": hist_hours + pred_df["Time"].tolist(),
        "Vehicles": hist_vals + pred_df["PredictedVolume"].tolist(),
        "Type": ["Historical"] * 6 + ["Predicted"] * len(pred_df),
    })
    fig2 = px.line(combined, x="Time", y="Vehicles", color="Type", markers=True,
                    color_discrete_map={"Historical": "#3b82f6", "Predicted": "#8b5cf6"})
    st.plotly_chart(style_fig(fig2), use_container_width=True)


# ============================================================
# PAGE: SETTINGS
# ============================================================
elif page == "⚙️ Settings":
    st.markdown("# Settings")
    st.caption("System configuration")

    tab1, tab2, tab3 = st.tabs(["🎯 Detection", "🚦 Signal Timing", "🔔 Alerts"])

    with tab1:
        st.markdown("#### Detection Parameters")
        st.slider("Confidence Threshold", 0.0, 1.0, 0.35, 0.05)
        st.slider("IOU Threshold", 0.0, 1.0, 0.45, 0.05)
        st.selectbox("YOLO Model", ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov8l.pt"])
        st.multiselect("Classes to Detect",
                        ["Car", "Truck", "Bus", "Motorcycle", "Bicycle", "Person", "Traffic Light", "Stop Sign"],
                        default=["Car", "Truck", "Bus", "Motorcycle", "Bicycle"])

    with tab2:
        st.markdown("#### Signal Timing Rules (seconds)")
        c1, c2, c3, c4 = st.columns(4)
        c1.number_input("Low density", value=20)
        c2.number_input("Medium density", value=40)
        c3.number_input("High density", value=60)
        c4.number_input("Very High density", value=90)
        st.number_input("Speed Limit (km/h)", value=60)
        st.number_input("Pixels per Meter (calibration)", value=8.0)

    with tab3:
        st.markdown("#### Alert Preferences")
        st.checkbox("Emergency vehicle sound alert", value=True)
        st.checkbox("Red light violation alerts", value=True)
        st.checkbox("Overspeed alerts", value=True)
        st.checkbox("Illegal parking alerts", value=False)
        st.text_input("Alert notification email")

    st.markdown("---")
    st.button("💾 Save Settings", type="primary")


# ============================================================
# PAGE: ABOUT
# ============================================================
elif page == "ℹ️ About":
    st.markdown("# About This System")
    st.markdown("""
    <div class="panel">
    <h4>AI-Based Intelligent Traffic Monitoring and Smart Traffic Signal Management System</h4>
    <p>A final-year Smart City project combining computer vision, machine learning, and
    real-time analytics to monitor traffic, manage adaptive signal timing, and detect
    violations automatically.</p>
    <p><b>Core Modules:</b> YOLOv8 Detection • ByteTrack Multi-Object Tracking • Density Classification
    • Adaptive Signal Control • Speed Estimation • Emergency Vehicle Priority • Violation Detection
    • ML-based Traffic Prediction</p>
    <p><b>Tech Stack:</b> Python, YOLOv8, OpenCV, Streamlit, SQLite, Plotly, Scikit-Learn</p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("#### System Architecture")
    st.graphviz_chart("""
    digraph {
        rankdir=LR;
        bgcolor="transparent";
        node [shape=box, style="rounded,filled", fillcolor="#131a2b", fontcolor="#e6e9f0", color="#3b82f6"];
        edge [color="#8b94ab"];
        Camera -> Detector [label="frames"];
        Detector -> Tracker;
        Tracker -> Density;
        Density -> SignalController;
        Tracker -> SpeedEstimator;
        Tracker -> ViolationModules;
        SignalController -> Dashboard;
        SpeedEstimator -> Dashboard;
        ViolationModules -> Dashboard;
        Density -> Dashboard;
        Dashboard -> Database;
        Database -> Prediction;
        Prediction -> Dashboard;
    }
    """)


# ============================================================
# FOOTER
# ============================================================
st.markdown('<div class="footer-tag">Smart Traffic Control System • Running on simulated data • v0.1</div>',
            unsafe_allow_html=True)

if auto_refresh:
    time.sleep(5)
    st.rerun()

Writing app.py


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true > streamlit_log.txt 2>&1 &
import time
time.sleep(8)
!cat streamlit_log.txt



2026-07-22 04:51:46.914 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.12.148.231:8501



In [ ]:
!curl -s http://localhost:8501 | head -20

<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />


In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) ^C


In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok

# Free account bana ke token lo: https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3GktcP6XTOZJGETHsH3DPDemhCs_5QpAybvjq6YUYurCZhq9H")

# Kill any existing tunnels
ngrok.kill()

public_url = ngrok.connect(8501)
print("🚀 Dashboard live at:", public_url)

🚀 Dashboard live at: NgrokTunnel: "https://secluded-profile-bride.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
import os
os.chdir("/content/TrafficMonitoringSystem")
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())

Current directory: /content/TrafficMonitoringSystem
Files here: ['density.py', 'screenshots', 'speed_estimator.py', 'videos', 'yolov8n.pt', 'traffic.db', 'reports', 'config.py', 'app.py', 'signal_controller.py', 'database.py', 'detector.py', 'models', 'output_annotated.mp4', '__pycache__', 'istockphoto-2278287172-640_adpp_is.mp4', 'emergency_vehicle.py', 'charts', 'tracker.py', 'streamlit_log.txt', 'utils.py']


In [ ]:
!kill -9 $(lsof -t -i:8501) 2>/dev/null
!streamlit run app.py --server.port 8501 --server.headless true > streamlit_log.txt 2>&1 &
import time
time.sleep(10)
!cat streamlit_log.txt



2026-07-22 04:54:30.572 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.12.148.231:8501

